# 00 — Môi trường thực thi & Khả năng tái lập (Environment & Reproducibility)

## 1. Mục tiêu
Notebook này thiết lập và kiểm toán môi trường tính toán phục vụ toàn bộ dự án **XAI-Football**.
Mục tiêu cụ thể:
- Thu thập metadata về hệ điều hành, phiên bản Python và các thư viện cốt lõi (PyTorch, PyG, scikit-learn, v.v.).
- Kiểm tra tính khả dụng của thiết bị tăng tốc phần cứng (CUDA/GPU).
- Xác minh commit Git HEAD và nhánh làm việc để đảm bảo khả năng truy xuất nguồn gốc (traceability).
- Tính mã băm SHA-256 của toàn bộ file cấu hình trong `configs/` nhằm phát hiện mọi thay đổi cấu hình ngoài ý muốn.
- Kiểm kê toàn diện sự tồn tại của các tập dữ liệu (`raw`, `interim`, `processed`), phân chia (`splits`), mô hình (`models`), và bảng kết quả (`results`).
- Xuất toàn bộ metadata kiểm toán ra file `results/metadata/official/environment.json`.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 0 — Kiểm toán Tiên quyết (Prerequisite Audit)**, được thực thi trước mọi bước tiền xử lý hay huấn luyện mô hình.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# Đảm bảo đường dẫn gốc của repository
# Gốc repo: chạy được cả khi kernel đứng ở notebooks/ (mặc định của Jupyter/nbconvert)
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs").is_dir())
assert (REPO_ROOT / "configs").exists(), "Vui lòng chạy notebook từ thư mục gốc của repository!"

## 3. Input và nguồn dữ liệu
- Môi trường Python runtime hiện tại.
- Các file cấu hình trong thư mục `configs/`.
- Cấu trúc thư mục dữ liệu và artifact: `data/`, `experiments/`, `models/`, `results/`.

## 4. Config/script/module sử dụng
- Các file YAML: `configs/data.yaml`, `configs/graph.yaml`, `configs/split.yaml`, `configs/train.yaml`.
- Module `xai_football.utils.environment` (cùng code với `scripts/00_generate_environment_metadata.py`),
  `xai_football.utils.provenance`.
- `requirements.lock.txt`: phiên bản thư viện đã khóa (pip freeze).

## 5. Các bước xử lý
### 5.1. Thu thập phiên bản thư viện và môi trường phần cứng

In [2]:
from xai_football.utils.environment import collect_environment_metadata

# Dùng CHUNG hàm với scripts/00_generate_environment_metadata.py để hai nơi ghi ra cùng một định dạng
env_info = collect_environment_metadata(REPO_ROOT)
pkg_versions = env_info["packages"]

df_env = pd.DataFrame(list(pkg_versions.items()), columns=["Thư viện", "Phiên bản"])
print(f"Hệ điều hành: {env_info['os']}")
print(f"Python:       {env_info['python_version']}")
print(f"CUDA Device:  {env_info['cuda_device_name']}")
display(df_env)

D:\dpkltn\XAI-sports-knowledge-discovery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\dpkltn\XAI-sports-knowledge-discovery\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Hệ điều hành: Windows-11-10.0.26200-SP0
Python:       3.13.0
CUDA Device:  CPU only


,Thư viện,Phiên bản
0,torch,2.14.1+cpu
1,torch_geometric,2.8.1
2,pandas,3.0.6
3,numpy,2.5.4
4,scipy,1.18.1
5,sklearn,1.9.1
6,matplotlib,3.11.2
7,seaborn,0.13.2
8,yaml,6.0.3
9,nbformat,5.11.1


### 5.2. Kiểm tra Git Branch & Commit SHA

In [3]:
git_info = env_info["git"]
print("Git Branch: ", git_info.get("branch"))
print("Commit SHA: ", git_info.get("commit_sha"))
print("Clean Tree: ", git_info.get("clean_working_tree"))

Git Branch:  main
Commit SHA:  fc18a929b82914cea6c9ff1bb54d89550597f12e
Clean Tree:  False


### 5.3. Băm SHA-256 các file cấu hình (Config Hash Audit)

In [4]:
config_hashes = env_info["config_hashes"]
df_configs = pd.DataFrame(list(config_hashes.items()), columns=["Config File", "SHA-256 Hash"])
display(df_configs)
print("SHA-256 requirements.lock.txt:", env_info.get("requirements_lock_sha256"))

,Config File,SHA-256 Hash
0,configs/data.yaml,00fdc1bdb7d3a5fd53f74b2844482c7769cffdf7488d2a...
1,configs/data_pilot.yaml,9f6df979afff6b8d160ac6a7620151aecbf8eba822d179...
2,configs/graph.yaml,068a38cdda6b0f8e349eb88f93d34ae3fb5b0976787020...
3,configs/hybrid_centrality.yaml,4c468f82b645a53d83fc9bf7bc04d8e9a7f6660a860920...
4,configs/model/baseline_tabular.yaml,497f740384feb5a4b97ab16903e37b46514c08a912be29...
5,configs/model/gatv2.yaml,eefff6218b25c1643229b98d68df87d2a39a6460fb201e...
6,configs/model/gcn.yaml,599fe80ac58e7cd51d3748f7382ec1240ddebf5d179181...
7,configs/model/graphsage.yaml,3ab90dd99d7d2b817b0cee1c4f4e674faab768e2c0f626...
8,configs/model/mpnn.yaml,e1b8533986eaf356259dc4f8b96156d11e5ec10a49e4ed...
9,configs/split.yaml,2e82c44e092ac5b74a74b0fc28f1e86ff3b3ef3e5c75c1...


SHA-256 requirements.lock.txt: a308244f39cab34f523dfa09b10a9c0481d4d6467034a8e2d1bb04a53a0c8fab


### 5.4. Kiểm kê Artifact thực tế (Artifact Inventory)

In [5]:
artifacts_to_check = [
    ("Raw Data Matches (PL 15/16)", REPO_ROOT / "data" / "raw" / "2" / "27"),
    ("Interim Possessions", REPO_ROOT / "data" / "interim" / "official" / "possessions.parquet"),
    ("Interim Passes", REPO_ROOT / "data" / "interim" / "official" / "possession_events.parquet"),
    ("Processed Graphs (.pt)", REPO_ROOT / "data" / "processed" / "official" / "graphs.pt"),
    ("Processed Graph Index", REPO_ROOT / "data" / "processed" / "official" / "graph_index.parquet"),
    ("Train Matches Split", REPO_ROOT / "data" / "splits" / "train_matches.csv"),
    ("Val Matches Split", REPO_ROOT / "data" / "splits" / "val_matches.csv"),
    ("Test Matches Split", REPO_ROOT / "data" / "splits" / "test_matches.csv"),
    ("Graph Split Summary", REPO_ROOT / "results" / "tables" / "official" / "graph_split_summary.csv"),
    ("Official Model Comparison", REPO_ROOT / "results" / "tables" / "official" / "model_comparison.csv"),
    ("Official Baseline Comparison", REPO_ROOT / "results" / "tables" / "official" / "baseline_comparison.csv"),
    ("Significance Tests", REPO_ROOT / "results" / "tables" / "official" / "significance_tests.csv"),
    ("Requirements Lock", REPO_ROOT / "requirements.lock.txt"),
] + [
    (f"{m.upper()} Best Checkpoint", REPO_ROOT / "models" / "official" / f"{m}_best.pt")
    for m in ("gcn", "mpnn", "gatv2", "graphsage")
]

inventory_rows = []
for label, path in artifacts_to_check:
    exists = path.exists()
    size_str = "N/A"
    if exists:
        if path.is_file():
            size_mb = path.stat().st_size / (1024 * 1024)
            size_str = f"{size_mb:.2f} MB"
        else:
            n_items = len(list(path.iterdir()))
            size_str = f"{n_items} files"
    inventory_rows.append({"Artifact": label, "Tồn tại": exists, "Đường dẫn": str(path.relative_to(REPO_ROOT)), "Kích thước": size_str})

df_inventory = pd.DataFrame(inventory_rows)
display(df_inventory)

,Artifact,Tồn tại,Đường dẫn,Kích thước
0,Raw Data Matches (PL 15/16),True,data\raw\2\27,760 files
1,Interim Possessions,True,data\interim\official\possessions.parquet,0.28 MB
2,Interim Passes,True,data\interim\official\possession_events.parquet,14.69 MB
3,Processed Graphs (.pt),True,data\processed\official\graphs.pt,94.10 MB
4,Processed Graph Index,True,data\processed\official\graph_index.parquet,0.40 MB
5,Train Matches Split,True,data\splits\train_matches.csv,0.00 MB
6,Val Matches Split,True,data\splits\val_matches.csv,0.00 MB
7,Test Matches Split,True,data\splits\test_matches.csv,0.00 MB
8,Graph Split Summary,True,results\tables\official\graph_split_summary.csv,0.00 MB
9,Official Model Comparison,True,results\tables\official\model_comparison.csv,0.00 MB


## 6. Kết quả và cách diễn giải
- Bảng trên liệt kê tình trạng sẵn sàng của các artifact chính.
- Nếu các artifact cốt lõi (`possessions.parquet`, `graphs.pt`, `graph_split_summary.csv`) đều tồn tại (`Tồn tại = True`), toàn bộ pipeline đã được chuẩn bị đầy đủ để tái lập các phân tích và biểu đồ trực quan.

## 7. Kiểm tra chất lượng (Quality Checks)

In [6]:
checks = []
checks.append({"Check": "Python Version >= 3.10", "Status": "PASS" if sys.version_info >= (3, 10) else "FAIL"})
checks.append({"Check": "PyTorch Installed", "Status": "PASS" if pkg_versions.get("torch") != "not installed" else "FAIL"})
checks.append({"Check": "PyG Installed", "Status": "PASS" if pkg_versions.get("torch_geometric") != "not installed" else "FAIL"})
checks.append({"Check": "Config Hashes Computed", "Status": "PASS" if len(config_hashes) >= 4 else "FAIL"})
checks.append({"Check": "requirements.lock.txt tồn tại", "Status": "PASS" if env_info.get("requirements_lock_sha256") else "FAIL"})
checks.append({"Check": "Mọi artifact Bước 0-6 tồn tại", "Status": "PASS" if df_inventory["Tồn tại"].all() else "FAIL"})

df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["Status"] == "PASS").all(), "Có kiểm tra chất lượng môi trường không đạt!"

,Check,Status
0,Python Version >= 3.10,PASS
1,PyTorch Installed,PASS
2,PyG Installed,PASS
3,Config Hashes Computed,PASS
4,requirements.lock.txt tồn tại,PASS
5,Mọi artifact Bước 0-6 tồn tại,PASS


## 8. Output được lưu ở đâu

In [7]:
out_meta_dir = REPO_ROOT / "results" / "metadata" / "official"
out_meta_dir.mkdir(parents=True, exist_ok=True)
env_out_path = out_meta_dir / "environment.json"

with open(env_out_path, "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=2, ensure_ascii=False)

print(f"Đã lưu thông tin môi trường vào: {env_out_path}")

Đã lưu thông tin môi trường vào: D:\dpkltn\XAI-sports-knowledge-discovery\results\metadata\official\environment.json


## 9. Cách tái chạy
Chạy lệnh sau từ thư mục gốc của repository:
```bash
pip install -e .
python -m jupyter nbconvert --to notebook --execute notebooks/00_environment_and_reproducibility.ipynb
```

## 10. Giới hạn và giả định
- Notebook phụ thuộc vào môi trường đã cài các gói trong `requirements.txt`.
- Kết quả kiểm tra thiết bị CUDA phụ thuộc vào máy của người dùng (nếu không có GPU NVIDIA, code tự động chuyển về CPU an toàn).